# Industry Prediction from Cyber Threat Intelligence Data

## Machine Learning Mini Project

### Project Overview

This project develops a machine learning classification system to predict the industry associated with a cybersecurity threat intelligence report.

The project covers the complete machine learning workflow:

- Dataset loading and exploration
- Data cleaning
- Target preparation
- Feature engineering
- TF-IDF text representation
- One-hot encoding
- Model training
- Model comparison
- Model evaluation
- Final model selection
- Model and preprocessing artifact saving

# 1. Problem Definition

## Objective

The objective of this project is to build a machine learning classification model that predicts the industry category associated with a cybersecurity threat intelligence report.

The dataset contains information such as report titles, descriptions, tags, malware families, attack IDs, and countries. These features are processed and converted into numerical representations that can be used by machine learning algorithms.

The final system is designed to accept relevant threat intelligence information and predict the most likely industry category.

In [72]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score
)

# 2. Dataset Loading

The dataset contains cybersecurity threat intelligence reports collected from the OTX threat intelligence data source.

The dataset includes information such as report title, description, tags, malware families, attack IDs, industries, countries, and other metadata.

In [73]:
df = pd.read_csv("/content/1_otx_threat_intel.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (2365, 14)


,Pulse_ID,Title,Description,Author,Created,Modified,TLP,Tags,Malware_Families,Attack_IDs,Industries,Countries,Indicators_Count,Subscribers
0,69f1e236e4e192f639298d53,Multi-Stage Malware Execution Chain Analysis,A sophisticated multi-stage malware execution ...,AlienVault,2026-04-29T10:49:26.327000,2026-04-29T10:50:57.999000,white,"payload extraction, c2 communication, defense ...",Unknown,"T1036.005, T1082, T1071, T1140, T1036, T1055, ...",Unknown,Unknown,0.0,0.0
1,69f1de85544538ce8b03332a,User interaction with a ClickFix-style phishin...,A ClickFix-style phishing campaign leveraged s...,AlienVault,2026-04-29T10:33:41.967000,2026-04-29T10:44:36.742000,white,"phishing, lumma stealer, powershell, informati...","HijackLoader, Lumma Stealer - S1213, LummaStealer","T1218.007, T1005, T1555, T1036, T1055, T1059, ...",Unknown,Unknown,0.0,0.0
2,69f1d20216d6091f01f8a6eb,Kyber ransomware is not just post-quantum name...,A detailed technical analysis confirms that Ky...,AlienVault,2026-04-29T09:40:17.996000,2026-04-29T10:14:38.724000,white,"post-quantum cryptography, x25519, aes-ctr enc...",Kyber,"T1489, T1135, T1082, T1112, T1070.001, T1222, ...","Defense, Technology",United States of America,0.0,0.0
3,69f1d26f3c7a8e098eccb448,Rebex-based Telegram RAT Targeting Vietnam,A sophisticated CHM-based malware campaign has...,AlienVault,2026-04-29T09:42:07.871000,2026-04-29T10:13:37.390000,white,"multi-stage payload, telegram rat, chm infecti...",Unknown,"T1053.005, T1036.005, T1204.002, T1497.001, T1...",Unknown,Unknown,0.0,0.0
4,69f1d2d45ec26fc5e1ca72f4,KYCShadow: An Android Banking Malware Exploiti...,An Android malware campaign masquerading as a ...,AlienVault,2026-04-29T09:43:48.542000,2026-04-29T10:12:57.758000,white,"india targeting, android banking trojan, otp t...",KYCShadow,Unknown,Finance,"British Indian Ocean Territory, India",0.0,0.0


# 3. Dataset Exploration

In [74]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2365 entries, 0 to 2364
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Pulse_ID          2365 non-null   object 
 1   Title             2365 non-null   object 
 2   Description       2364 non-null   object 
 3   Author            2365 non-null   object 
 4   Created           2365 non-null   object 
 5   Modified          2365 non-null   object 
 6   TLP               2365 non-null   object 
 7   Tags              2365 non-null   object 
 8   Malware_Families  2365 non-null   object 
 9   Attack_IDs        2365 non-null   object 
 10  Industries        2365 non-null   object 
 11  Countries         2365 non-null   object 
 12  Indicators_Count  2365 non-null   float64
 13  Subscribers       2365 non-null   float64
dtypes: float64(2), object(12)
memory usage: 258.8+ KB


In [75]:
df.columns.tolist()

['Pulse_ID',
 'Title',
 'Description',
 'Author',
 'Created',
 'Modified',
 'TLP',
 'Tags',
 'Malware_Families',
 'Attack_IDs',
 'Industries',
 'Countries',
 'Indicators_Count',
 'Subscribers']

categorical exploration

In [76]:
print(df['TLP'].value_counts())
print(df['Indicators_Count'].value_counts())
print(df['Subscribers'].value_counts())
print(df['Industries'].value_counts().head(10))
print(df['Countries'].value_counts().head(10))

TLP
white    2362
green       3
Name: count, dtype: int64
Indicators_Count
0.0    2365
Name: count, dtype: int64
Subscribers
0.0    2365
Name: count, dtype: int64
Industries
Unknown                1229
Finance                 116
Government              112
Technology               50
Finance, Technology      48
Government, Defense      43
Government, Finance      19
Defense, Government      17
Finance, Government      16
Retail                   15
Name: count, dtype: int64
Countries
Unknown                                  1306
United States of America                   90
Russian Federation                         78
Ukraine                                    43
China                                      37
British Indian Ocean Territory, India      33
Japan                                      23
Brazil                                     21
Taiwan                                     13
Israel                                     11
Name: count, dtype: int64


### Initial Observations

The dataset contains 2,365 records and 14 columns. The `Industries` column contains the target information used for classification.

A large number of records contain `Unknown` as their industry value. Therefore, these records are removed before preparing the final classification target.

The `Indicators_Count` and `Subscribers` columns contain constant zero values in the inspected dataset, so they do not provide useful variation for model training.

# 4. Data Cleaning

## Removing Unknown Target Values

Records with `Unknown` as the industry label cannot be used as meaningful supervised learning targets. Therefore, these records are removed.

## Handling Multi-Industry Labels

Some records contain multiple industries separated by commas. For this classification task, the first listed industry is selected as the main industry label.

## Grouping Rare Categories

To reduce the number of extremely rare classes, the ten most frequent industry categories are retained. All remaining categories are grouped into an `Other` class.

In [77]:
# Remove records with unknown industry labels
df_clean = df[df['Industries'] != 'Unknown'].copy()

# Select the first industry when multiple industries are listed
df_clean['Industries_main'] = df_clean['Industries'].apply(
    lambda x: str(x).split(',')[0].strip()
)

print("Rows after removing Unknown:", len(df_clean))
print(df_clean['Industries_main'].value_counts().head(10))

Rows after removing Unknown: 1136
Industries_main
Government            371
Finance               233
Technology            119
Manufacturing          52
Defense                49
Retail                 40
Energy                 37
Healthcare             36
Education              32
Telecommunications     30
Name: count, dtype: int64


In [78]:
top_categories = df_clean['Industries_main'].value_counts().head(10).index.tolist()

print("Top categories:", top_categories)

df_clean['Industries_final'] = df_clean['Industries_main'].apply(
    lambda x: x if x in top_categories else 'Other'
)

print(df_clean['Industries_final'].value_counts())

Top categories: ['Government', 'Finance', 'Technology', 'Manufacturing', 'Defense', 'Retail', 'Energy', 'Healthcare', 'Education', 'Telecommunications']
Industries_final
Government            371
Finance               233
Other                 137
Technology            119
Manufacturing          52
Defense                49
Retail                 40
Energy                 37
Healthcare             36
Education              32
Telecommunications     30
Name: count, dtype: int64


In [79]:
print("Duplicate rows:", df_clean.duplicated().sum())
print("Duplicate Pulse_IDs:", df_clean["Pulse_ID"].duplicated().sum())

Duplicate rows: 0
Duplicate Pulse_IDs: 15


In [80]:
df_final = df_clean.drop(columns=[
    'Indicators_Count', 'Subscribers', 'TLP',   # constant values
    'Industries', 'Industries_main',             # ab humare paas Industries_final hai
    'Pulse_ID'                                    # sirf ID hai, koi info nahi
])
print(df_final.columns.tolist())
print(df_final.shape)

['Title', 'Description', 'Author', 'Created', 'Modified', 'Tags', 'Malware_Families', 'Attack_IDs', 'Countries', 'Industries_final']
(1136, 10)


In [81]:
print(df_final['Tags'].head(5))
print("---")
print(df_final['Malware_Families'].value_counts().head(15))
print("---")
print(df_final['Countries'].value_counts().head(10))

2     post-quantum cryptography, x25519, aes-ctr enc...
4     india targeting, android banking trojan, otp t...
5     tax scams, bec, valleyrat, winos4.0, social en...
6     tor hidden service, covert persistence, spear-...
10    journalist targeting, uyghur targeting, govers...
Name: Tags, dtype: object
---
Malware_Families
Unknown                                                288
Cobalt Strike - S0154                                    9
BeaverTail, InvisibleFerret                              7
AsyncRAT                                                 5
Lumma Stealer                                            5
ROKRAT - S0240                                           5
Grandoreiro - S0531                                      3
ValleyRAT                                                3
Astaroth - S0373, Guildma                                3
Akira                                                    3
Crimson RAT                                              3
Cobalt Strike - S0154, P

In [82]:
# 1. Malware families ko binary feature banayein
df_final['has_known_malware'] = df_final['Malware_Families'].apply(
    lambda x: 0 if x == 'Unknown' else 1
)

# 2. Countries - pehla country nikalein
df_final['country_main'] = df_final['Countries'].apply(
    lambda x: x.split(',')[0].strip()
)
print(df_final['country_main'].value_counts().head(15))

# 3. Text length features (Description aur Tags se)
df_final['description_length'] = df_final['Description'].apply(lambda x: len(str(x)))
df_final['tags_count'] = df_final['Tags'].apply(lambda x: len(str(x).split(',')))

print(df_final[['has_known_malware', 'country_main', 'description_length', 'tags_count']].head())

country_main
Unknown                           339
United States of America          238
Russian Federation                 70
British Indian Ocean Territory     49
Ukraine                            44
China                              40
Brazil                             33
Japan                              18
Germany                            16
Taiwan                             13
Australia                          13
Belarus                            13
Argentina                          11
Canada                             11
Italy                              11
Name: count, dtype: int64
    has_known_malware                    country_main  description_length  \
2                   1        United States of America                 200   
4                   1  British Indian Ocean Territory                 200   
5                   1                         Unknown                 200   
6                   0                         Unknown                 200   
10    

In [66]:
print(len(df_final['Description'].iloc[0]))
print(df_final['description_length'].describe())

200
count    1136.000000
mean      297.359155
std        15.679147
min       178.000000
25%       300.000000
50%       300.000000
75%       300.000000
max       300.000000
Name: description_length, dtype: float64


In [83]:
top_countries = df_final['country_main'].value_counts().head(8).index.tolist()
df_final['country_final'] = df_final['country_main'].apply(
    lambda x: x if x in top_countries else 'Other'
)
print(df_final['country_final'].value_counts())

country_final
Unknown                           339
Other                             305
United States of America          238
Russian Federation                 70
British Indian Ocean Territory     49
Ukraine                            44
China                              40
Brazil                             33
Japan                              18
Name: count, dtype: int64


In [84]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Target
y = df_final['Industries_final'].reset_index(drop=True)

# Split the data FIRST to avoid data leakage
df_train, df_test = train_test_split(
    df_final,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Target values
y_train = df_train['Industries_final'].reset_index(drop=True)
y_test = df_test['Industries_final'].reset_index(drop=True)

# Country One-Hot Encoding
country_train = pd.get_dummies(
    df_train['country_final'],
    prefix='country'
)

country_test = pd.get_dummies(
    df_test['country_final'],
    prefix='country'
)

# Make test columns match training columns
country_test = country_test.reindex(
    columns=country_train.columns,
    fill_value=0
)

# Tags TF-IDF
# Fit ONLY on training data
tfidf = TfidfVectorizer(
    max_features=30,
    stop_words='english'
)

tags_train = tfidf.fit_transform(
    df_train['Tags'].astype(str)
)

tags_test = tfidf.transform(
    df_test['Tags'].astype(str)
)

tags_train_df = pd.DataFrame(
    tags_train.toarray(),
    columns=[
        f'tag_{w}'
        for w in tfidf.get_feature_names_out()
    ]
)

tags_test_df = pd.DataFrame(
    tags_test.toarray(),
    columns=[
        f'tag_{w}'
        for w in tfidf.get_feature_names_out()
    ]
)

# Numeric features
numeric_cols = [
    'has_known_malware',
    'description_length',
    'tags_count'
]

# Reset indexes before combining
X_train = pd.concat([
    df_train[numeric_cols].reset_index(drop=True),
    country_train.reset_index(drop=True),
    tags_train_df.reset_index(drop=True)
], axis=1)

X_test = pd.concat([
    df_test[numeric_cols].reset_index(drop=True),
    country_test.reset_index(drop=True),
    tags_test_df.reset_index(drop=True)
], axis=1)

print("Training feature matrix:", X_train.shape)
print("Testing feature matrix:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training feature matrix: (908, 42)
Testing feature matrix: (228, 42)
Training target: (908,)
Testing target: (228,)


In [85]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Model 1: Logistic Regression
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)

print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("F1 (weighted):", f1_score(y_test, y_pred_lr, average='weighted'))
print(classification_report(y_test, y_pred_lr))

# Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("=== Random Forest ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("F1 (weighted):", f1_score(y_test, y_pred_rf, average='weighted'))
print(classification_report(y_test, y_pred_rf))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== Logistic Regression ===
Accuracy: 0.2982456140350877
F1 (weighted): 0.34025713536549457
                    precision    recall  f1-score   support

           Defense       0.19      0.30      0.23        10
         Education       0.08      0.17      0.11         6
            Energy       0.09      0.29      0.13         7
           Finance       0.68      0.40      0.51        47
        Government       0.66      0.31      0.42        75
        Healthcare       0.05      0.14      0.08         7
     Manufacturing       0.08      0.10      0.09        10
             Other       0.24      0.18      0.20        28
            Retail       0.19      0.75      0.30         8
        Technology       0.60      0.25      0.35        24
Telecommunications       0.05      0.17      0.08         6

          accuracy                           0.30       228
         macro avg       0.26      0.28      0.23       228
      weighted avg       0.47      0.30      0.34       228

=== R

In [87]:
# Add Title TF-IDF to the final model

tfidf_title = TfidfVectorizer(
    max_features=20,
    stop_words='english'
)

# Fit Title TF-IDF only on training data
title_train = tfidf_title.fit_transform(
    df_train['Title'].astype(str)
)

# Transform test data using the training vocabulary
title_test = tfidf_title.transform(
    df_test['Title'].astype(str)
)

title_train_df = pd.DataFrame(
    title_train.toarray(),
    columns=[
        f'title_{w}'
        for w in tfidf_title.get_feature_names_out()
    ]
)

title_test_df = pd.DataFrame(
    title_test.toarray(),
    columns=[
        f'title_{w}'
        for w in tfidf_title.get_feature_names_out()
    ]
)

# Add title features to the existing train/test features
X_train2 = pd.concat(
    [X_train.reset_index(drop=True),
     title_train_df.reset_index(drop=True)],
    axis=1
)

X_test2 = pd.concat(
    [X_test.reset_index(drop=True),
     title_test_df.reset_index(drop=True)],
    axis=1
)

# Final Random Forest
rf2 = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight='balanced',
    random_state=42
)

rf2.fit(X_train2, y_train)

y_pred_rf2 = rf2.predict(X_test2)

print("=== Final Random Forest + Title TF-IDF ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf2))
print(
    "F1 (weighted):",
    f1_score(y_test, y_pred_rf2, average='weighted')
)
print(classification_report(y_test, y_pred_rf2))

=== Final Random Forest + Title TF-IDF ===
Accuracy: 0.4342105263157895
F1 (weighted): 0.40616716471539294
                    precision    recall  f1-score   support

           Defense       0.43      0.30      0.35        10
         Education       0.33      0.17      0.22         6
            Energy       0.00      0.00      0.00         7
           Finance       0.66      0.45      0.53        47
        Government       0.52      0.73      0.61        75
        Healthcare       0.17      0.14      0.15         7
     Manufacturing       0.14      0.10      0.12        10
             Other       0.21      0.21      0.21        28
            Retail       0.32      0.88      0.47         8
        Technology       0.31      0.17      0.22        24
Telecommunications       0.00      0.00      0.00         6

          accuracy                           0.43       228
         macro avg       0.28      0.29      0.26       228
      weighted avg       0.41      0.43      0.41  

In [88]:
import joblib

# Save final trained model
joblib.dump(rf2, 'final_model.pkl')

# Save preprocessing objects
joblib.dump(tfidf, 'tfidf_tags.pkl')
joblib.dump(tfidf_title, 'tfidf_title.pkl')

# Save the feature columns used by the final model
joblib.dump(list(X_train2.columns), 'feature_columns.pkl')

print("All model and preprocessing files saved successfully.")

All model and preprocessing files saved successfully.


In [89]:
import os

for file in [
    "final_model.pkl",
    "tfidf_tags.pkl",
    "tfidf_title.pkl",
    "feature_columns.pkl"
]:
    print(file, "→", "OK" if os.path.exists(file) else "MISSING")

final_model.pkl → OK
tfidf_tags.pkl → OK
tfidf_title.pkl → OK
feature_columns.pkl → OK
